# Contact coordinates: design (.oas) and SEM (.jpg)

Prints the contact centres from one `.oas` file and its matching SEM image.

| Output | Function | Units | (0, 0) is… | y points |
|---|---|---|---|---|
| Design `x_nm, y_nm` | `read_contacts`, or `read_contacts_tone_reversed` | nm | centre of the `.oas` = centre of the image | up |
| SEM `x_px, y_px` | `detect_contacts` (Otsu, region centroid) | px | centre of the top-left pixel | **down** |
| SEM `x_nm, y_nm` | `pixel_to_tile_nm`, pixel size = FOV / image width | nm | centre of the image | up |

Design `x_nm, y_nm` and SEM `x_nm, y_nm` are in the **same tile-local frame**, so they can be
compared directly. Global mask coordinates (tile centre from the metadata CSV) are not
applied yet.

Set the paths below and run all cells.

In [ ]:
OAS_PATH = r"path\to\tile.oas"
JPG_PATH = r"path\to\tile.jpg"
LAYER = (1, 0)         # (layer, datatype) of the contacts
CELL = None            # top cell name; None if the file has a single top cell
TONE_REVERSED = False  # True if the .oas draws the area AROUND the holes
FOV_NM = 2880.0        # SEM field of view (image width) in nm
DATABAR_ROWS = 0       # rows of SEM info bar at the bottom of the image to ignore
DARK_CONTACTS = True   # True if contacts look dark in the SEM image
MIN_AREA_PX = 20       # ignore regions smaller than this many pixels

In [ ]:
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.design import (
    list_layers,
    load_layout,
    read_contacts,
    read_contacts_tone_reversed,
)
from affine_ransac.io.sem_image import crop_databar, load_sem_image

## Design contacts (.oas)

In [ ]:
layout = load_layout(OAS_PATH)
print(f"Database unit: {layout.dbu * 1000:g} nm")
print(f"Layers: {list_layers(layout)}")

if TONE_REVERSED:
    centers_nm, sizes_nm = read_contacts_tone_reversed(
        layout, *LAYER, cell_name=CELL, frame_nm=(FOV_NM, FOV_NM))
else:
    centers_nm, sizes_nm = read_contacts(layout, *LAYER, cell_name=CELL)
design = pd.DataFrame({
    "x_nm": centers_nm[:, 0], "y_nm": centers_nm[:, 1],
    "w_nm": sizes_nm[:, 0], "h_nm": sizes_nm[:, 1],
})
print(f"{len(design)} contacts")
design

## SEM contacts (.jpg)

In [ ]:
image = crop_databar(load_sem_image(JPG_PATH), DATABAR_ROWS)
print(f"Image used: {image.shape[1]} x {image.shape[0]} px (after removing {DATABAR_ROWS} data-bar rows)")

pixel_size_nm = FOV_NM / image.shape[1]
print(f"Pixel size: {pixel_size_nm:.5f} nm")

found = detect_contacts(image, dark_contacts=DARK_CONTACTS, min_area_px=MIN_AREA_PX)
xy_nm = pixel_to_tile_nm(found.centers, image.shape, pixel_size_nm)
sem = pd.DataFrame({
    "x_px": found.centers[:, 0], "y_px": found.centers[:, 1],
    "x_nm": xy_nm[:, 0], "y_nm": xy_nm[:, 1], "area_px": found.areas,
})

print(f"Otsu threshold: {found.threshold:.0f}")
print(f"{len(sem)} contacts")
sem

## Design vs SEM in the tile-local frame

Both sets of centres on one plot. If the SEM points are mirrored or rotated relative to the
design, the image orientation differs from what we assume.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(design.x_nm, design.y_nm, marker="s", facecolors="none", edgecolors="tab:blue", label="design (.oas)")
ax.scatter(sem.x_nm, sem.y_nm, marker="+", color="tab:red", label="SEM")
ax.set_aspect("equal")
ax.set_xlabel("x (nm)")
ax.set_ylabel("y (nm)")
ax.set_title("Tile-local frame: (0, 0) = image centre, y up")
ax.legend()
plt.show()

## Visual check (optional)

Opens two interactive windows side by side: the design (polygons with centroid `+`) and the
SEM image (detected contours in green, centres as red `+`). The cell waits until you close
both windows.

In [ ]:
import pyqtgraph as pg
from affine_ransac import view_design, view_sem

app = pg.mkQApp()
design_win = view_design.build_window(OAS_PATH, LAYER, CELL, TONE_REVERSED, (FOV_NM, FOV_NM))
sem_win = view_sem.build_window(JPG_PATH, found)
design_win.show()
sem_win.show()
app.exec()